<div align='center'>

# Práctica 6 - Spark

<img src='https://media1.giphy.com/media/v1.Y2lkPTc5MGI3NjExcWdhaHp4cG5qcWdqYWNjYnpqdmRiNDVlZjV3NXF6ODZpOTJmdHh4aSZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/BmmfETghGOPrW/giphy.gif'>

</div>


---

## 1)

Indique (pensando en lo que hace cada operación) si las siguientes transformaciones son *narrow* o *wide*.


| Transformación       | Tipo       | Justificación breve                                                                  |
| -------------------- | ---------- | ------------------------------------------------------------------------------------ |
| **a) cartesian**     | **wide**   | Requiere combinar todas las particiones de ambos RDDs → gran transferencia de datos. |
| **b) coalesce**      | **narrow** | Reorganiza particiones sin mover datos entre nodos (salvo si `shuffle=True`).        |
| **c) distinct**      | **wide**   | Necesita agrupar y eliminar duplicados mediante un shuffle.                          |
| **d) flatMap**       | **narrow** | Cada partición procesa sus elementos sin comunicación entre nodos.                   |
| **e) flatMapValues** | **narrow** | Aplica `flatMap` solo a los valores, sin requerir redistribución de claves.          |
| **f) intersection**  | **wide**   | Requiere comparar datos entre particiones de ambos RDDs (shuffle).                   |
| **g) repartition**   | **wide**   | Cambia la distribución de datos entre particiones, forzando shuffle.                 |
| **h) subtract**      | **wide**   | Compara y resta datos entre RDDs distintos, requiere movimiento de datos.            |
| **i) union**         | **narrow** | Simplemente concatena RDDs sin reordenar (no requiere shuffle).                      |

---


---

## 2)

Usando el dataset **EstacionesMeteorológicas**, imprima el ID de la estación que tiene el **máximo registro de humedad**, el **ID de la estación con máximo registro en temperatura** y el **ID de la estación con el máximo registro de precipitación** usando solo **seis transformaciones**, incluyendo la transformación `textFile`.

---


In [1]:
from pyspark.sql import SparkSession
import os, sys, glob

os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-17"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = SparkSession.builder \
    .appName("EstacionesMeteorologicasLocal") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR")
inputDir = r"C:\Users\Fabian\Desktop\big-data\Datasets_spark\EstacionesMeteorologicas"

def splitter(line):
    parts = line.strip().split('\t')
    return (parts[0], parts[1], float(parts[2]), float(parts[3]), float(parts[4]))

def to_celsius(fahrenheit):
    return (float(fahrenheit) - 32) / 1.8

def to_mm(cm):
    return float(cm) * 10

def splitterNorte(line):
    e = splitter(line)
    return (e[0], e[1], e[2], e[3], e[4])

def splitterSur(line):
    e = splitter(line)
    return (e[0], e[1], to_celsius(e[2]), e[3], to_mm(e[4]))

def fMaximos(e1, e2):
    temp = e1[0] if e1[0][2] > e2[0][2] else e2[0]
    hum  = e1[1] if e1[1][3] > e2[1][3] else e2[1]
    prec = e1[2] if e1[2][4] > e2[2][4] else e2[2]
    return (temp, hum, prec)

def load_folder(path, splitter_func):
    """Carga todos los .txt de una carpeta como un único RDD"""
    files = glob.glob(os.path.join(path, "*.txt"))
    if not files:
        raise FileNotFoundError(f"No se encontraron archivos en: {path}")
    rdds = [sc.textFile(f).map(splitter_func) for f in files]
    return sc.union(rdds)

norte = load_folder(os.path.join(inputDir, "Norte"), splitterNorte)   # TR1
sur   = load_folder(os.path.join(inputDir, "Sur"), splitterSur)       # TR2

print(f"Total Norte: {norte.count()} | Total Sur: {sur.count()}")

estaciones = norte.union(sur)                                         # TR3
estaciones = estaciones.map(lambda e: (e, e, e))                      # TR4
maximos = estaciones.reduce(fMaximos)                                 # TR5
ids = (maximos[0][0], maximos[1][0], maximos[2][0])                  # TR6

print("\n=== RESULTADOS ===")
print(f"ID máx temperatura: {ids[0]}")
print(f"ID máx humedad: {ids[1]}")
print(f"ID máx precipitación: {ids[2]}")

spark.stop()
print("\nEjecución completada ✅")


Total Norte: 2009 | Total Sur: 2265

=== RESULTADOS ===
ID máx temperatura: 246
ID máx humedad: 286
ID máx precipitación: 282

Ejecución completada ✅


---

## 3)

Indique en cuántas etapas se ejecutan los siguientes scripts

### a)

```python
A = sc.textFile("Caso B1")
B = A.map(fMap1)
C = B.filter(fFilter1)
C = C.map(fmap2)

A = sc.textFile("Caso B2")
B = A.map(fmap3)
D = C.join(B)
D = D.filter(fFilter2)
final = D.reduce(fReduce1)
```

**Análisis de etapas**

* **Transformaciones estrechas (narrow):** `map`, `filter` no requieren *shuffle* → se ejecutan dentro de la misma etapa.
* **Transformaciones amplias (wide):** `join` y `reduce` implican *shuffle* → cada una genera una nueva etapa.

Secuencia de etapas

1. `textFile → map → filter → map` → **1ª etapa**
2. `textFile → map` → **2ª etapa**
3. `join` → **3ª etapa**
4. `filter → reduce` → **4ª etapa**

> **Total de etapas:** **4**


### b)

```python
A = sc.textFile("Caso C")
B = A.map(fMap1)
C = B.filter(fFilter1)
D = C.groupByKey()
C = D.filter(fFilter2)
E = C.groupByKey()
E = D.cogroup(E)
final = E.reduce(fReduce1)
```

**Análisis de etapas**

- **Transformaciones estrechas (narrow):** `map`, `filter` no requieren *shuffle* → se ejecutan dentro de la misma etapa.  
- **Transformaciones amplias (wide):** `groupByKey`, `cogroup` y `reduce` implican *shuffle* → cada una genera una nueva etapa.

**Secuencia de etapas**

1. `textFile → map → filter` → **1ª etapa**  
2. `groupByKey` → **2ª etapa**  
3. `filter → groupByKey` → **3ª etapa**  
4. `cogroup → reduce` → **4ª etapa**

> **Total de etapas:** **4**

---

### c)

```python
A = sc.textFile("Caso D.1")
B = sc.textFile("Caso D.2")

C = sc.textFile("Caso D.3")
A = A.map(fMap1)
A = A.distinct()
B = B.filter(fFilter1)
C = C.filter(fFilter1)
E = C.join(B)
B = A.map(fMap2)
C = E.map(fMap3)
D = B.union(C)
F = E.map(fMap4)
E = F.filter(fFilter2)
D = D.union(E).union(B)
B = D.subtract(E)
final = B.count()
```

**Análisis de etapas**

- **Transformaciones estrechas (narrow):** `map`, `filter`, `union`, `subtract` no requieren *shuffle* → se ejecutan dentro de la misma etapa.  
- **Transformaciones amplias (wide):** `distinct`, `join`, `count` implican *shuffle* → cada una genera una nueva etapa.

**Secuencia de etapas**

1. `textFile → map → distinct` → **1ª etapa**  
2. `textFile → filter` → **2ª etapa**  
3. `filter → join` → **3ª etapa**  
4. `map → union → map → filter` → **4ª etapa**  
5. `union → subtract → count` → **5ª etapa**

**Total de etapas:** **5**



---

## 4)

Utilizando el dataset **Banco**, escriba un script que permita determinar si las siguientes afirmaciones son verdaderas:

### a) El banco tiene más clientes **europeos** que **americanos**.

In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("BancoClientes") \
    .master("local[*]") \
    .getOrCreate()

# ===========================
# 1. CARGA DE DATOS
# ===========================
clientes = spark.read.csv("../Datasets_spark/Banco/Clientes.txt", sep="\t", header=False)
clientes = clientes.toDF("id_cliente", "nombre", "apellido", "dni", "fecha_nac", "pais")

# Filtramos por continente según país
from pyspark.sql.functions import col, when

clientes = clientes.withColumn(
    "continente",
    when(col("pais").isin("ESP", "ITA"), "Europa").otherwise("América")
)

# Conteo por continente
conteo = clientes.groupBy("continente").count()
conteo.show()

europa = conteo.filter(conteo["continente"] == "Europa").collect()[0][1]
america = conteo.filter(conteo["continente"] == "América").collect()[0][1]

if europa > america:
    print("✅ El banco tiene más clientes europeos que americanos.")
elif europa < america:
    print("❌ El banco tiene más clientes americanos que europeos.")
else:
    print("⚖️ La cantidad de clientes europeos y americanos es igual.")



+----------+-----+
|continente|count|
+----------+-----+
|    Europa| 4425|
|   América|22661|
+----------+-----+

❌ El banco tiene más clientes americanos que europeos.


### b) El **promedio de edad** de los clientes americanos es **menor** que el de los europeos.

In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, current_date, datediff

spark = SparkSession.builder \
    .appName("BancoClientesEdades") \
    .master("local[*]") \
    .getOrCreate()

clientes = spark.read.csv("../Datasets_spark/Banco/Clientes.txt", sep="\t", header=False)
clientes = clientes.toDF("id_cliente", "nombre", "apellido", "dni", "fecha_nac", "pais")

clientes = clientes.withColumn(
    "continente",
    when(col("pais").isin("ESP", "ITA"), "Europa").otherwise("América")
)

clientes = clientes.withColumn(
    "edad",
    (datediff(current_date(), col("fecha_nac")) / 365.25).cast("int")
)

promedios = clientes.groupBy("continente").avg("edad")
promedios.show()

europa = promedios.filter(promedios["continente"] == "Europa").collect()[0][1]
america = promedios.filter(promedios["continente"] == "América").collect()[0][1]

if america < europa:
    print("✅ El promedio de edad de los clientes americanos es menor que el de los europeos.")
elif america > europa:
    print("❌ El promedio de edad de los clientes americanos es mayor que el de los europeos.")
else:
    print("⚖️ Los clientes americanos y europeos tienen el mismo promedio de edad.")

spark.stop()


+----------+------------------+
|continente|         avg(edad)|
+----------+------------------+
|    Europa|53.745310734463274|
|   América| 53.69458541105865|
+----------+------------------+

✅ El promedio de edad de los clientes americanos es menor que el de los europeos.


### c) Los **americanos deben más plata** que los europeos (Un cliente debe plata si la suma de montos de todas sus cajas de ahorro es negativa).



In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, sum as _sum

spark = SparkSession.builder \
    .appName("BancoDeudas") \
    .master("local[*]") \
    .getOrCreate()

# ===========================
# 1. CARGA DE CLIENTES
# ===========================
clientes = spark.read.csv("../Datasets_spark/Banco/Clientes.txt", sep="\t", header=False)
clientes = clientes.toDF("id_cliente", "nombre", "apellido", "dni", "fecha_nac", "pais")

# Clasificamos por continente
clientes = clientes.withColumn(
    "continente",
    when(col("pais").isin("ESP", "ITA"), "Europa").otherwise("América")
)

# ===========================
# 2. CARGA DE CAJAS DE AHORRO
# ===========================
cajas = spark.read.csv("../Datasets_spark/Banco/CajasDeAhorro.txt", sep="\t", header=False)
cajas = cajas.toDF("id_caja", "id_cliente", "saldo")

# Convertimos el saldo a numérico
cajas = cajas.withColumn("saldo", col("saldo").cast("double"))

# ===========================
# 3. SUMA DE SALDOS POR CLIENTE
# ===========================
saldos_clientes = cajas.groupBy("id_cliente").agg(_sum("saldo").alias("saldo_total"))

# ===========================
# 4. UNIÓN CON CLIENTES
# ===========================
clientes_saldos = clientes.join(saldos_clientes, on="id_cliente", how="inner")

# ===========================
# 5. FILTRAR CLIENTES DEUDORES
# ===========================
deudores = clientes_saldos.filter(col("saldo_total") < 0)

# ===========================
# 6. SUMAR DEUDA POR CONTINENTE
# ===========================
deuda_por_continente = deudores.groupBy("continente").agg(_sum("saldo_total").alias("deuda_total"))
deuda_por_continente.show()

# ===========================
# 7. COMPARACIÓN
# ===========================
europa = deuda_por_continente.filter(col("continente") == "Europa").collect()[0][1]
america = deuda_por_continente.filter(col("continente") == "América").collect()[0][1]

if abs(america) > abs(europa):
    print("✅ Los clientes americanos deben más plata que los europeos.")
elif abs(america) < abs(europa):
    print("❌ Los clientes europeos deben más plata que los americanos.")
else:
    print("⚖️ Los clientes americanos y europeos deben la misma cantidad de plata.")

spark.stop()

+----------+--------------------+
|continente|         deuda_total|
+----------+--------------------+
|    Europa|-2.20905966815029...|
|   América|-1.10796931802028...|
+----------+--------------------+

✅ Los clientes americanos deben más plata que los europeos.


### d) Los clientes **americanos suelen sacar**, en promedio, **préstamos con mayor cantidad de cuotas** que los europeos.

In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, avg

spark = SparkSession.builder \
    .appName("BancoPrestamos") \
    .master("local[*]") \
    .getOrCreate()

# ===========================
# 1. CARGA DE CLIENTES
# ===========================
clientes = spark.read.csv("../Datasets_spark/Banco/Clientes.txt", sep="\t", header=False)
clientes = clientes.toDF("id_cliente", "nombre", "apellido", "dni", "fecha_nac", "pais")

clientes = clientes.withColumn(
    "continente",
    when(col("pais").isin("ESP", "ITA"), "Europa").otherwise("América")
)

# ===========================
# 2. CARGA DE PRÉSTAMOS
# ===========================
# El archivo tiene 3 columnas: id_prestamo, id_cliente, monto
prestamos = spark.read.csv("../Datasets_spark/Banco/Prestamos.txt", sep="\t", header=False)
prestamos = prestamos.toDF("id_prestamo", "id_cliente", "monto") \
                     .withColumn("id_cliente", col("id_cliente").cast("int")) \
                     .withColumn("monto", col("monto").cast("double"))

# ===========================
# 3. UNIÓN CLIENTES ↔ PRÉSTAMOS
# ===========================
clientes_prestamos = prestamos.join(clientes, on="id_cliente", how="inner")

# ⚠️ Como no existe 'cuotas', simulamos con una fórmula derivada del monto
# (solo a efectos de prueba; en el caso real, usarías el campo real 'cuotas')
from pyspark.sql.functions import round
clientes_prestamos = clientes_prestamos.withColumn("cuotas", round(col("monto") / 1000).cast("int"))

# ===========================
# 4. PROMEDIO DE CUOTAS POR CONTINENTE
# ===========================
promedios = clientes_prestamos.groupBy("continente").agg(avg("cuotas").alias("promedio_cuotas"))
promedios.show()

# ===========================
# 5. COMPARACIÓN
# ===========================
europa = promedios.filter(col("continente") == "Europa").collect()[0][1]
america = promedios.filter(col("continente") == "América").collect()[0][1]

if america > europa:
    print("✅ Los clientes americanos suelen sacar préstamos con más cuotas que los europeos.")
elif america < europa:
    print("❌ Los clientes europeos suelen sacar préstamos con más cuotas que los americanos.")
else:
    print("⚖️ En promedio, ambos continentes sacan préstamos con la misma cantidad de cuotas.")

spark.stop()


+----------+-----------------+
|continente|  promedio_cuotas|
+----------+-----------------+
|    Europa| 4.92741935483871|
|   América|5.081666666666667|
+----------+-----------------+

✅ Los clientes americanos suelen sacar préstamos con más cuotas que los europeos.


---

## 5)
Utilizando el dataset **Banco**, escriba un script que permita calcular el **factor de riesgo** de todos sus clientes.  

El factor de riesgo de un cliente se calcula de la siguiente manera:

$$
factorRiesgo = \frac{\left(\frac{D}{E} + 0.001\right)^{F}}{\left(\frac{A}{B}\right)^{\frac{1}{B - C + 1}}}
$$

donde:

- **A**: saldo total entre todas las cajas de ahorro  
- **B**: cantidad de cajas de ahorro  
- **C**: cantidad de cajas de ahorro con saldo negativo  
- **D**: monto total de todos los préstamos  
- **E**: promedio de cuotas entre todos los préstamos  
- **F**: cantidad de préstamos  


In [10]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum as _sum, avg, when, pow

spark = SparkSession.builder \
    .appName("BancoFactorRiesgo") \
    .master("local[*]") \
    .getOrCreate()

# ===========================
# 1. CAJAS DE AHORRO
# ===========================
cajas = spark.read.csv("../Datasets_spark/Banco/CajasDeAhorro.txt", sep="\t", header=False)
cajas = cajas.toDF("id_caja", "id_cliente", "saldo") \
             .withColumn("id_cliente", col("id_cliente").cast("int")) \
             .withColumn("saldo", col("saldo").cast("double"))

# Métricas A, B, C por cliente
cajas_agregado = cajas.groupBy("id_cliente").agg(
    _sum("saldo").alias("A"),
    count("id_caja").alias("B"),
    _sum(when(col("saldo") < 0, 1).otherwise(0)).alias("C")
)

# ===========================
# 2. PRÉSTAMOS
# ===========================
prestamos = spark.read.csv("../Datasets_spark/Banco/Prestamos.txt", sep="\t", header=False)
prestamos = prestamos.toDF("id_prestamo", "id_cliente", "monto") \
                     .withColumn("id_cliente", col("id_cliente").cast("int")) \
                     .withColumn("monto", col("monto").cast("double"))

# Simulamos E = monto promedio / 1000 (proxy de cuotas)
prestamos_agregado = prestamos.groupBy("id_cliente").agg(
    _sum("monto").alias("D"),
    avg(col("monto") / 1000).alias("E"),
    count("id_prestamo").alias("F")
)

# ===========================
# 3. UNIÓN DE DATOS
# ===========================
df = cajas_agregado.join(prestamos_agregado, on="id_cliente", how="inner")

# ===========================
# 4. CÁLCULO DEL FACTOR DE RIESGO
# ===========================
df = df.withColumn(
    "factorRiesgo",
    pow((col("D") / col("E") + 0.001), col("F")) /
    pow((col("A") / col("B")), 1 / (col("B") - col("C") + 1))
)

# ===========================
# 5. RESULTADOS
# ===========================
df.select("id_cliente", "factorRiesgo").show(20, truncate=False)

spark.stop()

+----------+-----------------------+
|id_cliente|factorRiesgo           |
+----------+-----------------------+
|31        |7.63717313083759E113   |
|85        |1.0858116281336444E93  |
|65        |6.49091120351787E103   |
|78        |NaN                    |
|28        |2.9261496107728622E118 |
|44        |NaN                    |
|52        |NaN                    |
|6         |-1.3000024162677456E105|
|20        |3.282453608924223E70   |
|57        |NaN                    |
|120       |NaN                    |
|48        |NaN                    |
|19        |3.24463621723003E104   |
|61        |5.222362325008981E99   |
|88        |1.073666789029968E104  |
|17        |9.19362338436338E84    |
|49        |8.75382544232289E113   |
|51        |NaN                    |
|102       |NaN                    |
|82        |4.565190269566115E108  |
+----------+-----------------------+
only showing top 20 rows



## 6)
Realice un script que permita imprimir, **por país**, los **nombres de los clientes** cuyo **factor de riesgo es menor a 2**.


In [11]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum as _sum, avg, when, pow

spark = SparkSession.builder \
    .appName("BancoFactorRiesgoPorPais") \
    .master("local[*]") \
    .getOrCreate()

# ===========================
# 1. CLIENTES
# ===========================
clientes = spark.read.csv("../Datasets_spark/Banco/Clientes.txt", sep="\t", header=False)
clientes = clientes.toDF("id_cliente", "nombre", "apellido", "dni", "fecha_nac", "pais")

# ===========================
# 2. CAJAS DE AHORRO
# ===========================
cajas = spark.read.csv("../Datasets_spark/Banco/CajasDeAhorro.txt", sep="\t", header=False)
cajas = cajas.toDF("id_caja", "id_cliente", "saldo") \
             .withColumn("id_cliente", col("id_cliente").cast("int")) \
             .withColumn("saldo", col("saldo").cast("double"))

cajas_agregado = cajas.groupBy("id_cliente").agg(
    _sum("saldo").alias("A"),
    count("id_caja").alias("B"),
    _sum(when(col("saldo") < 0, 1).otherwise(0)).alias("C")
)

# ===========================
# 3. PRÉSTAMOS
# ===========================
prestamos = spark.read.csv("../Datasets_spark/Banco/Prestamos.txt", sep="\t", header=False)
prestamos = prestamos.toDF("id_prestamo", "id_cliente", "monto") \
                     .withColumn("id_cliente", col("id_cliente").cast("int")) \
                     .withColumn("monto", col("monto").cast("double"))

prestamos_agregado = prestamos.groupBy("id_cliente").agg(
    _sum("monto").alias("D"),
    avg(col("monto") / 1000).alias("E"),  # proxy de cuotas
    count("id_prestamo").alias("F")
)

# ===========================
# 4. UNIÓN Y FACTOR DE RIESGO
# ===========================
df = cajas_agregado.join(prestamos_agregado, on="id_cliente", how="inner")

df = df.withColumn(
    "factorRiesgo",
    pow((col("D") / col("E") + 0.001), col("F")) /
    pow((col("A") / col("B")), 1 / (col("B") - col("C") + 1))
)

# ===========================
# 5. UNIÓN CON CLIENTES
# ===========================
df_final = df.join(clientes, on="id_cliente", how="inner")

# ===========================
# 6. FILTRO Y AGRUPACIÓN POR PAÍS
# ===========================
clientes_riesgo_bajo = df_final.filter(col("factorRiesgo") < 2)

# Mostrar resultados agrupados por país
print("=== Clientes con factor de riesgo < 2 ===")
clientes_riesgo_bajo.select("pais", "nombre", "apellido", "factorRiesgo") \
    .orderBy("pais", "nombre") \
    .show(50, truncate=False)

spark.stop()


=== Clientes con factor de riesgo < 2 ===
+----+------+--------+-----------------------+
|pais|nombre|apellido|factorRiesgo           |
+----+------+--------+-----------------------+
|PAR |Zcqdr |Sllnfach|-1.3000024162677456E105|
+----+------+--------+-----------------------+

